In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA_DIR = Path.cwd()
if not (DATA_DIR / "u.data").exists():
    DATA_DIR = Path.cwd() / "Popularity_Based_ML100K"
assert (DATA_DIR / "u.data").exists(), "Keep notebook with u.data, u.item and u.genre"
print("Dataset folder:", DATA_DIR)

Dataset folder: C:\HOPE AI\AI TAMIL COURSE\RECOMMENDATION SYSTEM\3.Popularity Based


## 1. Load the ML-100K data

In [2]:
ratings = pd.read_csv(DATA_DIR / "u.data", sep="\t", header=None, names=["user_id", "movie_id", "rating", "timestamp"])
genre_map = pd.read_csv(DATA_DIR / "u.genre", sep="|", header=None, names=["genre", "index"], encoding="latin-1")
genres = genre_map.loc[genre_map.genre != "unknown", "genre"].tolist()
columns = ["movie_id", "title", "release_date", "video_release_date", "IMDb_URL", "unknown"] + genres
movies = pd.read_csv(DATA_DIR / "u.item", sep="|", header=None, names=columns, encoding="latin-1")
print("Ratings:", ratings.shape, "Movies:", movies.shape, "Named genres:", len(genres))
display(movies[["movie_id", "title"] + genres[:4]].head())

Ratings: (100000, 4) Movies: (1682, 24) Named genres: 18


,movie_id,title,Action,Adventure,Animation,Children's
0,1,Toy Story (1995),0,0,1,1
1,2,GoldenEye (1995),1,1,0,0
2,3,Four Rooms (1995),0,0,0,0
3,4,Get Shorty (1995),1,0,0,0
4,5,Copycat (1995),0,0,0,0


## 2. Basic data checks

In [3]:
print("Missing ratings:", ratings[["user_id", "movie_id", "rating"]].isna().sum().to_dict())
print("Duplicate user-movie ratings:", ratings.duplicated(["user_id", "movie_id"]).sum())
print("Rating values:", sorted(ratings.rating.unique()))
print("Genres:", ", ".join(genres))

Missing ratings: {'user_id': 0, 'movie_id': 0, 'rating': 0}
Duplicate user-movie ratings: 0
Rating values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Genres: Action, Adventure, Animation, Children's, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western


## 3. Count ratings and calculate average rating

In [4]:
stats = ratings.groupby("movie_id").agg(rating_count=("rating", "size"), average_rating=("rating", "mean")).reset_index()
movie_stats = movies.merge(stats, on="movie_id", how="left")
movie_stats["rating_count"] = movie_stats["rating_count"].fillna(0).astype(int)
movie_stats["average_rating"] = movie_stats["average_rating"].fillna(0)
display(movie_stats[["title", "rating_count", "average_rating"]].sort_values("rating_count", ascending=False).head(10))

,title,rating_count,average_rating
49,Star Wars (1977),583,4.358491
257,Contact (1997),509,3.803536
99,Fargo (1996),508,4.155512
180,Return of the Jedi (1983),507,4.007890
293,Liar Liar (1997),485,3.156701
285,"English Patient, The (1996)",481,3.656965
287,Scream (1996),478,3.441423
0,Toy Story (1995),452,3.878319
299,Air Force One (1997),431,3.631090
120,Independence Day (ID4) (1996),429,3.438228


## 4. Weighted popularity score
Formula: **score = (v/(v+m)) × R + (m/(v+m)) × C**

- `v`: number of ratings for the movie
- `R`: average rating of the movie
- `C`: global mean rating
- `m`: minimum-rating threshold (60th percentile of rating counts among rated movies)

This Bayesian-style score balances quality and confidence. The threshold is applied **within each genre** for fair comparisons, with a fallback if a genre has fewer than five qualifying movies.

In [5]:
C = ratings.rating.mean()
m = float(stats.rating_count.quantile(0.60))
def weighted_score(v, R, m=m, C=C):
    return (v / (v + m)) * R + (m / (v + m)) * C
movie_stats["popularity_score"] = weighted_score(movie_stats.rating_count, movie_stats.average_rating)
print(f"Global mean C = {C:.3f}; threshold m = {m:.1f}")

Global mean C = 3.530; threshold m = 43.0


## 5. Top 5 movies for each of the 18 genres
A movie can belong to multiple genres and therefore appear in multiple lists. Unrated movies are excluded. Sort by weighted score, then rating count, then movie title.

In [6]:
def top_movies_by_genre(genre, top_n=5):
    if genre not in genres:
        raise ValueError(f"Unknown genre: {genre}. Choose from {genres}")
    candidates = movie_stats.loc[(movie_stats[genre] == 1) & (movie_stats.rating_count > 0)].copy()
    eligible = candidates.loc[candidates.rating_count >= m]
    if len(eligible) >= top_n:
        candidates = eligible
    candidates = candidates.sort_values(["popularity_score", "rating_count", "title"], ascending=[False, False, True]).head(top_n)
    result = candidates[["movie_id", "title", "rating_count", "average_rating", "popularity_score"]].copy()
    result.insert(0, "rank", range(1, len(result) + 1))
    result.insert(0, "genre", genre)
    return result.reset_index(drop=True)

all_top5 = pd.concat([top_movies_by_genre(g, 5) for g in genres], ignore_index=True)
assert len(genres) == 18
assert all_top5.groupby("genre").size().eq(5).all(), "Expected 5 movies in every genre"
print("Genres:", len(genres), "Total top-five rows:", len(all_top5))
display(all_top5.round(3))

Genres: 18 Total top-five rows: 90


,genre,rank,movie_id,title,rating_count,average_rating,popularity_score
0,Action,1,50,Star Wars (1977),583,4.358,4.302
1,Action,2,127,"Godfather, The (1972)",413,4.283,4.212
2,Action,3,174,Raiders of the Lost Ark (1981),420,4.252,4.185
3,Action,4,313,Titanic (1997),350,4.246,4.167
4,Action,5,172,"Empire Strikes Back, The (1980)",367,4.204,4.134
...,...,...,...,...,...,...,...
85,Western,1,661,High Noon (1952),88,4.102,3.914
86,Western,2,435,Butch Cassidy and the Sundance Kid (1969),216,3.949,3.879
87,Western,3,510,"Magnificent Seven, The (1954)",121,3.942,3.834
88,Western,4,203,Unforgiven (1992),182,3.868,3.803


## 6. Display each genre separately

In [7]:
for genre in genres:
    print(f"\n========== {genre.upper()} — TOP 5 ==========")
    display(top_movies_by_genre(genre)[["rank", "title", "rating_count", "average_rating", "popularity_score"]].round(3))


========== ACTION — TOP 5 ==========


,rank,title,rating_count,average_rating,popularity_score
0,1,Star Wars (1977),583,4.358,4.302
1,2,"Godfather, The (1972)",413,4.283,4.212
2,3,Raiders of the Lost Ark (1981),420,4.252,4.185
3,4,Titanic (1997),350,4.246,4.167
4,5,"Empire Strikes Back, The (1980)",367,4.204,4.134



========== ADVENTURE — TOP 5 ==========


,rank,title,rating_count,average_rating,popularity_score
0,1,Star Wars (1977),583,4.358,4.302
1,2,Raiders of the Lost Ark (1981),420,4.252,4.185
2,3,"Empire Strikes Back, The (1980)",367,4.204,4.134
3,4,"Princess Bride, The (1987)",324,4.173,4.098
4,5,Lawrence of Arabia (1962),173,4.231,4.092



========== ANIMATION — TOP 5 ==========


...

,rank,title,rating_count,average_rating,popularity_score
0,1,"Usual Suspects, The (1995)",267,4.386,4.267
1,2,Rear Window (1954),209,4.388,4.241
2,3,"Close Shave, A (1995)",112,4.491,4.224
3,4,"Silence of the Lambs, The (1991)",390,4.290,4.214
4,5,North by Northwest (1959),179,4.285,4.139



========== WAR — TOP 5 ==========


,rank,title,rating_count,average_rating,popularity_score
0,1,Schindler's List (1993),298,4.466,4.348
1,2,Casablanca (1942),243,4.457,4.317
2,3,Star Wars (1977),583,4.358,4.302
3,4,"Empire Strikes Back, The (1980)",367,4.204,4.134
4,5,Dr. Strangelove or: How I Learned to Stop Worr...,194,4.253,4.121



========== WESTERN — TOP 5 ==========


,rank,title,rating_count,average_rating,popularity_score
0,1,High Noon (1952),88,4.102,3.914
1,2,Butch Cassidy and the Sundance Kid (1969),216,3.949,3.879
2,3,"Magnificent Seven, The (1954)",121,3.942,3.834
3,4,Unforgiven (1992),182,3.868,3.803
4,5,"Good, The Bad and The Ugly, The (1966)",137,3.861,3.782


## 7. Interactive genre selection

In [8]:
selected_genre = "Action"  # Change to Comedy, Drama, Sci-Fi, etc.
display(top_movies_by_genre(selected_genre).round(3))

,genre,rank,movie_id,title,rating_count,average_rating,popularity_score
0,Action,1,50,Star Wars (1977),583,4.358,4.302
1,Action,2,127,"Godfather, The (1972)",413,4.283,4.212
2,Action,3,174,Raiders of the Lost Ark (1981),420,4.252,4.185
3,Action,4,313,Titanic (1997),350,4.246,4.167
4,Action,5,172,"Empire Strikes Back, The (1980)",367,4.204,4.134


## 8. Export recommendations to CSV and Excel

In [9]:
all_top5.to_csv(DATA_DIR / "Popularity_Top5_All_18_Genres.csv", index=False)
with pd.ExcelWriter(DATA_DIR / "Popularity_Top5_All_18_Genres.xlsx") as writer:
    all_top5.to_excel(writer, sheet_name="All 18 Genres", index=False)
    for genre in genres:
        top_movies_by_genre(genre).to_excel(writer, sheet_name=genre.replace("/", "-")[:31], index=False)
print("Saved CSV and Excel files in", DATA_DIR)

Saved CSV and Excel files in C:\HOPE AI\AI TAMIL COURSE\RECOMMENDATION SYSTEM\3.Popularity Based
